## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual [here](https://quantra.quantinsti.com/quantra-notebook).
2. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
3. Before running this notebook on your local PC:
   - You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on **"Run Codes Locally on Your Machine"** in the course.
   - You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.

# GTT & AMO

Welcome! So far, we've focused on orders that are active for a single trading day. But what about longer-term strategies? What if you want to set an order that remains active for weeks or months, or place trades after the market has closed?

For these scenarios, we use strategic order types. In this notebook, you will learn:

-   What a **Good Till Triggered (GTT)** order is and how to use it for long-term entries and exits.
-   How to place **After Market Orders (AMO)** to have your trades ready for the next session.

### 1. Authentication

As always, we'll start by authenticating our Kite Connect client.

In [ ]:
from kiteconnect import KiteConnect
import logging

logging.basicConfig(level=logging.INFO)

# --- Credentials & Initialisation ---
api_key = "YOUR_API_KEY"
access_token = "YOUR_ACCESS_TOKEN"

try:
    kite = KiteConnect(api_key=api_key)
    kite.set_access_token(access_token)
    profile = kite.profile()
    print(f"Logged in as {profile['user_name']}")
    logging.info("Authentication successful!")
except Exception as e:
    logging.error(f"Authentication failed: {e}")

INFO:root:Authentication successful!


### 2. Good Till Triggered (GTT)

A GTT is an instruction you leave with Zerodha that stays active for up to one year. When your specified price (the trigger price) is met, Zerodha places a Limit order on the exchange on your behalf. This is perfect for 'set it and forget it' strategies.

We use the special `place_gtt()` method for this, not `place_order()`.

#### Single-Leg GTT

This is a GTT with a single trigger for a single action. A classic use case is buying a stock on a significant price dip.

Let's create a GTT to buy 5 shares of TCS if the price ever falls to or below ₹3000.

In [ ]:
try:
    # Fetch current market price
    current_ltp = kite.ltp('NSE:TCS')['NSE:TCS']['last_price']
    print(f"Current LTP for TCS: {current_ltp}")

    # Define the orders to be placed when triggered
    orders = [
        {
            "exchange": kite.EXCHANGE_NSE,
            "tradingsymbol": "TCS",
            "transaction_type": kite.TRANSACTION_TYPE_BUY,
            "quantity": 5,
            "order_type": kite.ORDER_TYPE_LIMIT,
            "product": kite.PRODUCT_CNC,  # Required parameter
            "price": 3001.0
        }
    ]

    # Create the GTT trigger
    trigger_data = kite.place_gtt(
        trigger_type=kite.GTT_TYPE_SINGLE,
        tradingsymbol='TCS',
        exchange=kite.EXCHANGE_NSE,        # Use constant instead of string
        trigger_values=[3000.0],           # Trigger when price <= 3000
        last_price=current_ltp,            # Current market price for reference
        orders=orders                      # The orders to execute when triggered
    )

    print(f"Single-Leg GTT placed successfully. Trigger ID: {trigger_data['trigger_id']}")

except Exception as e:
    print(f"Error placing GTT: {e}")

Error placing GTT: Insufficient permission for that call.


#### One-Cancels-Other (OCO) GTT

An OCO GTT lets you set two triggers on an existing holding: a target above the current price and a stop-loss below it. If one is triggered, the other is automatically cancelled. It's like a long-term Bracket Order for your portfolio.

Let's assume we own 10 shares of HDFCBANK and want to set a target at ₹2300 and a stop-loss at ₹1700.

In [ ]:
try:
    current_ltp = kite.ltp('NSE:HDFCBANK')['NSE:HDFCBANK']['last_price']

    orders = [
        {
            "exchange": kite.EXCHANGE_NSE,
            "tradingsymbol": "HDFCBANK",
            "transaction_type": kite.TRANSACTION_TYPE_SELL,
            "quantity": 5,
            "order_type": kite.ORDER_TYPE_LIMIT,
            "product": kite.PRODUCT_CNC,
            "price": 2300.0  # Target price
        },
        {
            "exchange": kite.EXCHANGE_NSE,
            "tradingsymbol": "HDFCBANK",
            "transaction_type": kite.TRANSACTION_TYPE_SELL,
            "quantity": 5,
            "order_type": kite.ORDER_TYPE_LIMIT,
            "product": kite.PRODUCT_CNC,
            "price": 1700.0  # Stop-loss price
        }
    ]

    trigger_data = kite.place_gtt(
        trigger_type=kite.GTT_TYPE_OCO,
        tradingsymbol='HDFCBANK',
        exchange=kite.EXCHANGE_NSE,
        trigger_values=[1700.0, 2300.0],  #  stop-loss and Target triggers
        last_price=current_ltp,
        orders=orders
    )

    print(f"OCO GTT placed successfully. Trigger ID: {trigger_data['trigger_id']}")

except Exception as e:
    print(f"Error placing OCO GTT: {e}")

OCO GTT placed successfully. Trigger ID: 280473012


### 3. After Market Orders (AMO)

An After Market Order (AMO) allows you to place orders after the market has closed for the day. The broker collects these orders and sends them to the exchange for execution when the market opens on the next trading day.

To place an AMO, we use the standard `place_order` method but set `variety='amo'`.

In [ ]:
# Placing an After Market Order (AMO)
try:
    order_id = kite.place_order(
        variety='amo',
        exchange=kite.EXCHANGE_NSE,
        tradingsymbol='INFY',
        transaction_type=kite.TRANSACTION_TYPE_BUY,
        quantity=5,
        product=kite.PRODUCT_CNC, # For delivery
        order_type=kite.ORDER_TYPE_LIMIT,
        price=1600.0
    )
    print(f"AMO placed successfully. Order ID: {order_id}")
except Exception as e:
    print(f"Error placing AMO: {e}")

AMO placed successfully. Order ID: 250716201100217


### 4. Conclusion

You are now equipped with powerful tools for longer-term and strategic trading:
-   **GTT** orders allow you to set triggers that can last for up to a year, perfect for swing and positional strategies.
-   **AMO** orders provide the convenience of placing your trades after market hours, ready for the next session.

In the next video, we will cover our final advanced order type, Iceberg orders, which are designed to help execute large trades with minimal market impact.<br><br>